# STAF (Spatio-Temporal Attention Framework) Google Colab Training Notebook

This self-contained notebook automates the setup, preprocessing, training, and evaluation of the **STAF Multimodal Deepfake Detector** baseline model on Google Colab using GPU acceleration.

### Workflow
1. Mount Google Drive (for saving checkpoints persistently)
2. Clone the Git Repository
3. Install dependencies & verify GPU (CUDA)
4. Set up the FakeAVCeleb dataset
5. Run GPU-accelerated audio-visual preprocessing
6. Train the baseline model (with AMP and optimized loaders)
7. Evaluate the checkpoint on the test set
8. Backup checkpoints and plots to Google Drive

## Step 1: Mount Google Drive
Mount your Google Drive to persistently save model checkpoints, training logs, and evaluation plots.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone the Git Repository
Clone the official STAF repository from GitHub and switch to the project directory.

In [ ]:
!git clone https://github.com/RAJ-VARUN13/staf-forensics-multimodal.git
%cd staf-forensics-multimodal

## Step 3: Install Dependencies & Verify GPU
Install the required libraries and verify that Colab has allocated a CUDA-enabled GPU.

In [ ]:
# Install python requirements
!pip install -r requirements.txt

# Verify PyTorch can see the GPU
import torch
cuda_available = torch.cuda.is_available()
print("CUDA GPU Available:", cuda_available)
if cuda_available:
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Training will be slow. Switch runtime to GPU under: Runtime -> Change runtime type")

## Step 4: Locate or Upload FakeAVCeleb Dataset
Upload the FakeAVCeleb dataset zip file to Google Drive, then run the cell below to extract it directly into Colab's fast local SSD storage (`/content`).

Update `zip_path` to point to the location of the ZIP file on your Google Drive.

In [ ]:
import os

# Path to the FakeAVCeleb zip archive on your Google Drive
zip_path = "/content/drive/MyDrive/FakeAVCeleb_v1.2.zip"
extract_dir = "/content/FakeAVCeleb"

if os.path.exists(zip_path):
    print(f"Extracting dataset from {zip_path}...")
    os.makedirs(extract_dir, exist_ok=True)
    !unzip -q "{zip_path}" -d /content/
    print("Dataset extraction complete!")
else:
    print(f"ZIP file not found at {zip_path}.")
    print("Please upload FakeAVCeleb_v1.2.zip to your Google Drive and verify the path.")

## Step 5: Run GPU-Accelerated Preprocessing
Since we are running on a CUDA GPU, face detection and cropping will run extremely fast (typically <0.02s per frame instead of 17s on CPU). Runs the full preprocessing pipeline:
1. Extract video frames
2. Detect faces (RetinaFace)
3. Align & Crop faces
4. Extract audio (WAV)
5. Sync modalities & build splits (train/val/test)

In [ ]:
# 1. Extract frames
!python -m staf.preprocessing.extract_frames --config staf/configs/colab.yaml

# 2. Run face detection
!python -m staf.preprocessing.detect_faces --config staf/configs/colab.yaml

# 3. Crop and align detected faces
!python -m staf.preprocessing.crop_faces --config staf/configs/colab.yaml

# 4. Extract audio tracks
!python -m staf.preprocessing.extract_audio --config staf/configs/colab.yaml

# 5. Synchronize modalities and build subject-independent splits
!python -m staf.preprocessing.sync_modalities --config staf/configs/colab.yaml

## Step 6: Train the Baseline Multimodal Detector
Run baseline model training using the Colab environment overrides (`staf/configs/colab.yaml`).
AMP (mixed precision) and 4-worker persistent DataLoaders are enabled automatically.

In [ ]:
# Run training
!python train.py --config staf/configs/colab.yaml --no_wandb

## Step 7: Evaluate the Best Checkpoint
Locate the saved best model checkpoint and evaluate it on the test set split to generate metrics and plots.

In [ ]:
import glob

# Automatically find the latest experiment run results directory
runs = sorted(glob.glob("/content/results/*_colab_baseline_run"))
if runs:
    latest_run = runs[-1]
    checkpoint_path = f"{latest_run}/checkpoints/best_model.pt"
    print(f"Evaluating latest model checkpoint: {checkpoint_path}")
    !python evaluate.py --checkpoint "{checkpoint_path}" --split test
else:
    print("No training results found in /content/results/. Did the training complete successfully?")

## Step 8: Backup Results & Checkpoints to Google Drive
Move checkpoints, evaluation plots, and metrics files to Google Drive for persistent access.

In [ ]:
import shutil
from pathlib import Path

gdrive_dest = "/content/drive/MyDrive/STAF_experiments"
os.makedirs(gdrive_dest, exist_ok=True)

if os.path.exists("/content/results"):
    print(f"Backing up training results to Google Drive ({gdrive_dest})...")
    !cp -r /content/results/* "{gdrive_dest}/"
    print("Backup complete! Your checkpoints, logs, and plots are safely stored in your Drive.")
else:
    print("No results found to backup.")